# Notebook 2 — M1 (Retinopathy) + M2 (Acanthosis Nigricans)
## Explainable Multimodal Diabetes/Metabolic Risk Framework

**This notebook:**
1. Fine-tunes ResNet18 on APTOS 2019 for 5-class DR severity (M1)
2. Trains HAM10000 skin lesion model then fine-tunes binary AN proxy (M2)
3. Attaches Grad-CAM to both models
4. Extracts per-sample risk scores and saves to Drive
5. Generates all M1/M2 Section 4 figures

**Runtime:** ~90 min on Colab T4 GPU for both modules

In [ ]:
# ─── Setup ────────────────────────────────────────────────────────────────────
!pip install -q torch torchvision pillow tqdm opencv-python scikit-learn matplotlib

import os, sys
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR  = '/content/drive/MyDrive/MultimodalDisease'
REPO_DIR  = '/content/Multimodal_Disease'  # upload your repo here
os.environ['MMDISEASE_BASE'] = BASE_DIR
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Set seeds
from src.utils.data_utils import set_seed
set_seed(42)

## MODULE 1 — Diabetic Retinopathy Detection (APTOS 2019)

In [ ]:
# ─── M1: Load APTOS DataLoaders ──────────────────────────────────────────────
from src.utils.data_utils import load_aptos_dataloaders
from src.config import M1

APTOS_DIR = f'{BASE_DIR}/data/aptos2019'
train_loader, val_loader, test_loader, test_df = load_aptos_dataloaders(
    aptos_dir=APTOS_DIR, batch_size=M1['batch_size'], num_workers=2
)

In [ ]:
# ─── M1: Visualize class distribution ────────────────────────────────────────
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(f'{APTOS_DIR}/train.csv')
class_names = ['No DR (0)', 'Mild (1)', 'Moderate (2)', 'Severe (3)', 'Proliferative (4)']

fig, ax = plt.subplots(figsize=(10, 5))
counts = df['diagnosis'].value_counts().sort_index()
colors = ['#55A868', '#4C72B0', '#DD8452', '#C44E52', '#8172B2']
bars = ax.bar(class_names, counts.values, color=colors, alpha=0.85)
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(count), ha='center', fontsize=10)
ax.set_title('APTOS 2019 — DR Severity Distribution')
ax.set_ylabel('Number of Images')
plt.tight_layout()
plt.savefig(f'{BASE_DIR}/outputs/figures/m1_class_distribution.png', dpi=150)
plt.show()

In [ ]:
# ─── M1: Train RetinopathyModel ───────────────────────────────────────────────
from src.models.m1_retinopathy import train_retinopathy_model

SAVE_PATH = f'{BASE_DIR}/saved_models/m1_retinopathy.pth'

m1_model, m1_history = train_retinopathy_model(
    train_loader, val_loader,
    device       = device,
    epochs       = M1['epochs'],
    lr           = M1['lr'],
    save_path    = SAVE_PATH
)

In [ ]:
# ─── M1: Training curves ─────────────────────────────────────────────────────
from src.utils.viz_utils import plot_training_curves

plot_training_curves(
    m1_history['train_losses'], m1_history['val_losses'],
    m1_history['train_accs'],   m1_history['val_accs'],
    module_name='M1 Retinopathy', save=True
)

In [ ]:
# ─── M1: Evaluate on test set ────────────────────────────────────────────────
import numpy as np
from src.models.m1_retinopathy import extract_risk_scores, severity_to_binary
from src.utils.eval_utils import compute_metrics, print_classification_report
from src.utils.viz_utils import plot_confusion_matrix, plot_roc_curve

m1_risks, m1_preds, m1_labels = extract_risk_scores(m1_model, test_loader, device)

# Save risk scores for M5
np.save(f'{BASE_DIR}/outputs/scores/m1_risk_scores.npy', m1_risks)
np.save(f'{BASE_DIR}/outputs/scores/m1_labels.npy', m1_labels)
print(f'M1 risk scores saved. Shape: {m1_risks.shape}')

# Metrics (binary: DR present vs absent)
m1_binary_preds  = severity_to_binary(m1_preds)
m1_binary_labels = severity_to_binary(m1_labels)

print('\n=== M1 Metrics (Binary: DR Present vs Absent) ===')
m1_metrics = compute_metrics(
    m1_binary_labels, m1_binary_preds,
    y_prob=m1_risks, module_name='M1 Retinopathy'
)

# 5-class confusion matrix
plot_confusion_matrix(m1_labels, m1_preds, class_names, 'M1 Retinopathy (5-class)', save=True)

# ROC curve (binary)
plot_roc_curve(m1_binary_labels, m1_risks, 'M1 Retinopathy', save=True)

In [ ]:
# ─── M1: Grad-CAM visualization ──────────────────────────────────────────────
from src.xai.gradcam import GradCAM, compute_gradcam, batch_gradcam
from src.utils.viz_utils import overlay_gradcam

# Get Grad-CAM target layer (layer4 of ResNet18)
target_layer = m1_model.layer4[-1]

# Generate Grad-CAM for 8 test samples
cam_results = batch_gradcam(
    model        = m1_model,
    dataloader   = test_loader,
    target_layer = target_layer,
    device       = device,
    n_samples    = 8
)

# Plot first 4 examples
for i, result in enumerate(cam_results[:4]):
    import cv2
    heatmap_resized = cv2.resize(result['heatmap'], (224, 224))
    fig = overlay_gradcam(
        original_img = result['image'],
        heatmap      = heatmap_resized,
        title        = f'M1 Grad-CAM — True: {class_names[result["label"]]}, Pred: {class_names[result["pred"]]}',
        save         = True,
        filename     = f'm1_gradcam_sample_{i}'
    )
    plt.show()

print('M1 Grad-CAM figures saved.')

## MODULE 2 — Acanthosis Nigricans Detection (HAM10000 Proxy)

In [ ]:
# ─── M2: Load HAM10000 DataLoaders (Stage 1 — 7-class pretraining) ───────────
from src.utils.data_utils import load_ham_dataloaders, make_binary_acanthosis_dataset
from src.config import M2

HAM_DIR = f'{BASE_DIR}/data/ham10000'

ham_train_loader, ham_val_loader, ham_test_loader = load_ham_dataloaders(
    ham_dir=HAM_DIR, batch_size=M2['batch_size'], num_workers=2
)

In [ ]:
# ─── M2 Stage 1: Pretrain on HAM10000 7-class ────────────────────────────────
from src.models.m2_acanthosis import train_ham_pretrain

HAM_MODEL_PATH = f'{BASE_DIR}/saved_models/m2_ham_pretrained.pth'

ham_model = train_ham_pretrain(
    train_loader = ham_train_loader,
    val_loader   = ham_val_loader,
    device       = device,
    epochs       = 15,
    lr           = 1e-4,
    save_path    = HAM_MODEL_PATH
)
print('HAM10000 Stage 1 pretraining complete.')

In [ ]:
# ─── M2 Stage 2: Binary AN proxy fine-tuning ─────────────────────────────────
from src.models.m2_acanthosis import train_acanthosis_model

an_train, an_val, an_test = make_binary_acanthosis_dataset(
    ham_dir=HAM_DIR, batch_size=M2['batch_size'], num_workers=2
)

AN_MODEL_PATH = f'{BASE_DIR}/saved_models/m2_acanthosis.pth'

m2_model, m2_history = train_acanthosis_model(
    train_loader     = an_train,
    val_loader       = an_val,
    device           = device,
    epochs           = M2['epochs'],
    lr               = M2['lr'],
    ham_weights_path = HAM_MODEL_PATH,
    save_path        = AN_MODEL_PATH
)
print('M2 Acanthosis proxy training complete.')

In [ ]:
# ─── M2: Training curves ─────────────────────────────────────────────────────
plot_training_curves(
    m2_history['train_losses'], m2_history['val_losses'],
    m2_history['train_accs'],   m2_history['val_accs'],
    module_name='M2 Acanthosis', save=True
)

In [ ]:
# ─── M2: Evaluate + save scores ──────────────────────────────────────────────
from src.models.m2_acanthosis import extract_risk_scores as m2_extract

m2_risks, m2_preds, m2_labels = m2_extract(m2_model, an_test, device)

np.save(f'{BASE_DIR}/outputs/scores/m2_risk_scores.npy', m2_risks)
np.save(f'{BASE_DIR}/outputs/scores/m2_labels.npy', m2_labels)
print(f'M2 risk scores saved. Shape: {m2_risks.shape}')

print('\n=== M2 Metrics ===')
m2_metrics = compute_metrics(
    m2_labels, m2_preds,
    y_prob=m2_risks, module_name='M2 Acanthosis'
)

plot_confusion_matrix(m2_labels, m2_preds, ['Normal', 'AN-Proxy'], 'M2 Acanthosis', save=True)
plot_roc_curve(m2_labels, m2_risks, 'M2 Acanthosis', save=True)

In [ ]:
# ─── M2: Grad-CAM visualization ──────────────────────────────────────────────
m2_target_layer = m2_model.layer4[-1]

m2_cam_results = batch_gradcam(
    model        = m2_model,
    dataloader   = an_test,
    target_layer = m2_target_layer,
    device       = device,
    n_samples    = 4
)

for i, result in enumerate(m2_cam_results):
    heatmap_r = cv2.resize(result['heatmap'], (224, 224))
    label_str = 'AN-Proxy' if result['label'] == 1 else 'Normal'
    pred_str  = 'AN-Proxy' if result['pred']  == 1 else 'Normal'
    overlay_gradcam(
        original_img = result['image'],
        heatmap      = heatmap_r,
        title        = f'M2 Grad-CAM — True: {label_str}, Pred: {pred_str}',
        save         = True,
        filename     = f'm2_gradcam_sample_{i}'
    )
print('M2 Grad-CAM figures saved.')

In [ ]:
# ─── Summary: M1 + M2 metrics ────────────────────────────────────────────────
from src.utils.eval_utils import build_results_table

all_metrics = [m1_metrics, m2_metrics]
table = build_results_table(all_metrics)
print('\n=== M1 + M2 Results Table ===')
print(table.to_string())

from src.utils.viz_utils import plot_module_metrics_bar
plot_module_metrics_bar([m1_metrics, m2_metrics], save=True)

print('\nNotebook 2 complete. Models and scores saved to Drive.')
print(f'  M1 model: {BASE_DIR}/saved_models/m1_retinopathy.pth')
print(f'  M2 model: {BASE_DIR}/saved_models/m2_acanthosis.pth')
print(f'  M1 scores: {BASE_DIR}/outputs/scores/m1_risk_scores.npy')
print(f'  M2 scores: {BASE_DIR}/outputs/scores/m2_risk_scores.npy')